# Wan 2.2 (5B) ile ücretsiz video üretimi: Kaggle / Colab

**Önerilen ortam: Kaggle** (haftalık ücretsiz GPU kotası, ~29 GB RAM).
Colab'ın ücretsiz sürümünde RAM yaklaşık 12 GB; model yüklenirken RAM yetmeyip oturum çökebilir.

Kaggle ayarları (sağ panel → *Settings*):
1. **Accelerator:** `GPU T4 x2` (veya P100)
2. **Internet:** açık (telefon doğrulaması gerekir)
3. **Persistence:** isteğe bağlı

Beklenti: T4'te 480p, 5 saniyelik bir klip **birkaç dakikadan uzun** sürebilir. İlk çalıştırmada
model (~20+ GB) indirilir. Bu notebook GPU'da test edilmedi; hata alırsan hata mesajını Claude'a yapıştır.

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

In [ ]:
import torch, gc, json, os, time
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - GPU açmayı unutma!")
print("bf16 destekli:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## 1. Ayarlar
`ORAN`: `9:16` (Reels/TikTok), `16:9` (YouTube), `1:1` (kare).
`KALITE`: `hizli` (480p, 20 adım, deneme için) veya `iyi` (daha yüksek çözünürlük, çok daha yavaş).

In [ ]:
ORAN = "9:16"
KALITE = "hizli"
SURE_SN = 5            # Wan 2.2 5B 24 fps üretir; 5 sn = 121 kare
TOHUM = 42             # Aynı tohum = aynı sonuç. Farklı deneme için değiştir.

COZUNURLUK = {
    "hizli": {"9:16": (832, 480), "16:9": (480, 832), "1:1": (640, 640)},
    "iyi":   {"9:16": (1280, 704), "16:9": (704, 1280), "1:1": (960, 960)},
}
YUKSEKLIK, GENISLIK = COZUNURLUK[KALITE][ORAN]
ADIM = 20 if KALITE == "hizli" else 40
FPS = 24
KARE = (SURE_SN * FPS // 4) * 4 + 1   # Wan kare sayısı 4k+1 olmalı
NEGATIF = ("blurry, low quality, distorted product, warped text, extra fingers, deformed hands, "
           "watermark, flicker, jitter, oversaturated, static frame")
print(f"{GENISLIK}x{YUKSEKLIK}, {KARE} kare, {ADIM} adım")

## 2. Modeli yükle (yazıdan video)

In [ ]:
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

MODEL = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()   # VRAM'e sığması için parçaları sırayla GPU'ya alır
try:
    pipe.vae.enable_tiling()      # VAE çözümlemesinde bellek tasarrufu
except Exception as e:
    print("VAE tiling yok:", e)

## 3. Tek klip üret

In [ ]:
PROMPT = ("hero shot of a sleek glass bottle of cold brew coffee with condensation droplets on a pedestal, "
          "Camera: slow 180-degree orbit, macro lens. Lighting: bright high-key studio lighting with colored rim lights. "
          "Style: dynamic commercial, vibrant colors, photorealistic.")

def uret(prompt, dosya, negatif=NEGATIF, tohum=TOHUM):
    basla = time.time()
    uretec = torch.Generator(device="cpu").manual_seed(tohum)
    kareler = pipe(prompt=prompt, negative_prompt=negatif, height=YUKSEKLIK, width=GENISLIK,
                   num_frames=KARE, num_inference_steps=ADIM, guidance_scale=5.0,
                   generator=uretec).frames[0]
    export_to_video(kareler, dosya, fps=FPS)
    print(f"{dosya} hazır ({(time.time() - basla) / 60:.1f} dk)")
    return dosya

uret(PROMPT, "deneme.mp4")
from IPython.display import Video
Video("deneme.mp4", embed=True, width=320)

## 4. Storyboard'daki tüm sahneleri toplu üret
Bilgisayarında `python araclar/plan_uret.py brief.json -k proje` çalıştır, çıkan **`sahneler.json`**
dosyasını Kaggle'a yükle (*Add Input → Upload*) veya Colab'da dosyalar paneline sürükle, yolunu aşağıya yaz.
Kota bitmesin diye `SADECE` ile tek tek sahne de seçebilirsin.

In [ ]:
SAHNELER_JSON = "sahneler.json"     # Kaggle'da örn: /kaggle/input/<veri-adi>/sahneler.json
SADECE = None                       # Örn: [1, 3] sadece 1. ve 3. sahne

with open(SAHNELER_JSON, encoding="utf-8") as f:
    plan = json.load(f)
os.makedirs("sahneler", exist_ok=True)
for sahne in plan["sahneler"]:
    if SADECE and sahne["no"] not in SADECE:
        continue
    print(f"--- {sahne['no']}. {sahne['ad']} ---")
    uret(sahne["prompt"], os.path.join("sahneler", sahne["dosya"]), negatif=sahne.get("negatif", NEGATIF))

In [ ]:
# Hepsini tek dosyada indir (Kaggle: sağ panel Output; Colab: dosyalar paneli)
!cd sahneler && zip -q -r ../sahneler.zip . && ls -la ../sahneler.zip

## 5. Görselden video (ürün fotoğrafın varsa bunu kullan)
Ürünün şekli/etiketi yazıdan üretimde bozulur; **kendi ürün fotoğrafından** başlatınca korunur.
Önce yazıdan video modelini bellekten sil (aşağıdaki hücre yapıyor), sonra fotoğrafını yükleyip yolunu yaz.

In [ ]:
del pipe; gc.collect(); torch.cuda.empty_cache()

from diffusers import WanImageToVideoPipeline
from diffusers.utils import load_image

vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe_i2v = WanImageToVideoPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=DTYPE)
pipe_i2v.enable_model_cpu_offload()
try:
    pipe_i2v.vae.enable_tiling()
except Exception as e:
    print("VAE tiling yok:", e)

In [ ]:
from IPython.display import Video
GORSEL = "urun.jpg"   # yüklediğin ürün fotoğrafı
PROMPT_I2V = "the product slowly rotates, soft light sweeps across it, condensation droplets glisten, commercial look"

gorsel = load_image(GORSEL)
# Fotoğrafı hedef orana kırp
hedef_oran = GENISLIK / YUKSEKLIK
g, y = gorsel.size
if g / y > hedef_oran:
    yeni_g = int(y * hedef_oran); gorsel = gorsel.crop(((g - yeni_g) // 2, 0, (g + yeni_g) // 2, y))
else:
    yeni_y = int(g / hedef_oran); gorsel = gorsel.crop((0, (y - yeni_y) // 2, g, (y + yeni_y) // 2))
gorsel = gorsel.resize((GENISLIK, YUKSEKLIK))

basla = time.time()
kareler = pipe_i2v(image=gorsel, prompt=PROMPT_I2V, negative_prompt=NEGATIF, height=YUKSEKLIK, width=GENISLIK,
                   num_frames=KARE, num_inference_steps=ADIM, guidance_scale=5.0,
                   generator=torch.Generator(device="cpu").manual_seed(TOHUM)).frames[0]
export_to_video(kareler, "gorselden.mp4", fps=FPS)
print(f"hazır ({(time.time() - basla) / 60:.1f} dk)")
Video("gorselden.mp4", embed=True, width=320)

## Sorun giderme
- **CUDA out of memory:** `KALITE = "hizli"`, `SURE_SN = 3` yap; olmazsa oturumu yeniden başlat.
- **RAM yetmedi / oturum çöktü (Colab):** Kaggle'a geç.
- **Görüntü siyah / NaN:** P100'de bf16 yok; T4 seç.
- **Çok yavaş:** Normal. Ücretsiz GPU'da 5 sn klip dakikalar sürer. Önce `ADIM = 12` ile prompt dene, beğenince yükselt.